# Logistic Regression

Despite the name, logistic regression is a **classification** model. It fits a linear score (exactly like linear regression) and then squashes that score through the **sigmoid** function into a probability between 0 and 1. Thresholding that probability (usually at 0.5) turns it into a class label.

So the pipeline is: `features -> weighted sum -> sigmoid -> probability -> threshold -> class`.

**Topics covered in this notebook**

1. Intuition
2. Training on a dataset
3. Predicted probabilities
4. When to use it

## 1. Intuition

The model first computes a **linear score** (a weighted sum of the features plus a bias), just like linear regression:

$$z = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b = \mathbf{w}^\top \mathbf{x} + b$$

That score `z` can be any real number, but a probability has to live in $(0, 1)$. The **sigmoid** (logistic) function does exactly that mapping:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Key behaviour of the sigmoid:

- $z = 0 \Rightarrow \sigma(z) = 0.5$ (the decision point),
- large positive $z \Rightarrow \sigma(z) \to 1$ (confident class 1),
- large negative $z \Rightarrow \sigma(z) \to 0$ (confident class 0).

A probability above 0.5 is predicted as class 1, below 0.5 as class 0. The cell below evaluates the sigmoid at a few scores so you can see the S-shaped squash in action.

In [ ]:
import numpy as np  # vectorised math for the sigmoid

def sigmoid(z):
    # sigma(z) = 1 / (1 + e^-z): maps any real number into the open interval (0, 1),
    # so the raw linear score z can be read as a probability.
    return 1 / (1 + np.exp(-z))

# Sweep a few scores from very negative to very positive to see the S-curve:
# negative z -> near 0, z = 0 -> exactly 0.5, positive z -> near 1.
for z in [-4, -1, 0, 1, 4]:
    print(f"z={z:2d} -> probability {sigmoid(z):.3f}")

## 2. Training on a Dataset

We classify tumours as **malignant** vs **benign** using scikit-learn's built-in breast-cancer dataset (30 numeric features per tumour, no download needed).

**Why scale the features first?** The model works on the weighted sum $\mathbf{w}^\top \mathbf{x}$. In this dataset `mean area` runs into the thousands while `mean smoothness` is around 0.1 - a raw weighted sum would be dominated by the large-valued features, and the gradient-based solver would converge slowly (or hit `max_iter` and warn). **Standardizing** each feature to zero mean and unit variance,

$$x' = \frac{x - \mu}{\sigma},$$

puts every feature on a comparable footing so no single one dominates and the optimiser converges cleanly. We fit the scaler on the **training set only** and reuse those statistics on the test set, so no test information leaks into preprocessing.

**What is the model minimizing?** Logistic regression is trained to minimize the **log-loss** (binary cross-entropy), averaged over the $m$ training samples:

$$\mathcal{L} = -\frac{1}{m}\sum_{i=1}^{m}\Big[\, y_i \log \hat{y}_i + (1 - y_i)\log(1 - \hat{y}_i)\,\Big]$$

where $\hat{y}_i = \sigma(\mathbf{w}^\top \mathbf{x}_i + b)$. This loss punishes confident-but-wrong predictions very hard, which is what pushes the weights toward good probabilities.

In [ ]:
from sklearn.datasets import load_breast_cancer          # built-in labelled dataset
from sklearn.model_selection import train_test_split     # hold-out split
from sklearn.preprocessing import StandardScaler         # feature standardization (z-score)
from sklearn.linear_model import LogisticRegression      # the model itself
from sklearn.metrics import accuracy_score, precision_score, recall_score  # evaluation metrics
import matplotlib.pyplot as plt                          # plotting
import pandas as pd                                      # tabular view of the features

In [ ]:
cancer = load_breast_cancer()   # returns a Bunch (dict-like), not a plain tuple
# X: (569, 30) feature matrix; y: (569,) labels where 0 = malignant, 1 = benign.
X, y = cancer.data, cancer.target

In [ ]:
# Wrap the features in a DataFrame purely for a readable, labelled view of the data.
# (The model is trained on the raw NumPy array X, not on this DataFrame.)
df = pd.DataFrame(data=cancer.data, columns=cancer.feature_names)
df.head()

In [ ]:
# The 30 feature names: each tumour is described by the mean, standard error, and
# "worst" (largest) value of ten cell-nucleus measurements (radius, texture, ...).
cancer.feature_names

In [ ]:
# Hold out 20% of the data for testing. random_state fixes the shuffle so the split
# (and therefore every number below) is reproducible on every run.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Eyeball the raw data on its first two features (mean radius vs mean texture),
# coloured by the true class. The classes already separate fairly well here, which
# hints that a linear model can do a good job.
fig, ax = plt.subplots(figsize=(8, 6))   # own figure + axes for robust, isolated plotting
scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', edgecolor='k', alpha=0.7)
ax.set_title('Breast Cancer Dataset: Radius vs Texture')
ax.set_xlabel('mean radius')
ax.set_ylabel('mean texture')
fig.colorbar(scatter, ax=ax, label='Target (0 = Malignant, 1 = Benign)')
plt.show()

In [ ]:
# The RAW training features, before scaling. Notice the wildly different magnitudes
# across columns (areas in the hundreds/thousands, smoothness near 0.1) - this is
# exactly why we standardize in the next step.
X_train

In [ ]:
# Standardize so every feature is on a comparable scale: x' = (x - mu) / sigma.
# Fit the scaler on TRAIN only (learns each column's mu and sigma)...
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)   # apply to train
# ...then reuse the SAME train statistics on the test set - never re-fit on test,
# otherwise information from the test set would leak into preprocessing.
X_test_s = scaler.transform(X_test)

In [ ]:
# The SCALED training features: every column now has ~zero mean and ~unit variance,
# so values sit in a comparable range (roughly -3 to +3). Compare to the raw array above.
X_train_s

In [ ]:
# max_iter=1000 gives the solver plenty of steps to converge on the log-loss minimum.
# (With scaled features it converges well within this budget, so no ConvergenceWarning.)
model = LogisticRegression(max_iter=1000)
model.fit(X_train_s, y_train)          # learn weights w and bias b by minimizing log-loss
preds = model.predict(X_test_s)        # hard 0/1 labels for the held-out test set

In [ ]:
# In this dataset class 0 = malignant, which is the clinically important case to
# catch, so we report precision/recall for the malignant class explicitly.
# (By default these metrics use pos_label=1, i.e. benign.)
#   accuracy  = fraction of all predictions that are correct.
#   precision = of the tumours we flagged malignant, how many truly were.
#   recall    = of the truly malignant tumours, how many we caught (missing one is costly).
print("accuracy             :", round(accuracy_score(y_test, preds), 3))
print("precision (malignant):", round(precision_score(y_test, preds, pos_label=0), 3))
print("recall    (malignant):", round(recall_score(y_test, preds, pos_label=0), 3))

## 3. Predicted Probabilities

Unlike a bare classifier that only emits a label, logistic regression gives you a **calibrated-ish probability** for each class via `predict_proba`. Each row sums to 1:

$$\big[\; P(\text{class }0 \mid \mathbf{x}),\;\; P(\text{class }1 \mid \mathbf{x}) \;\big], \qquad P(\text{class }1) = \sigma(\mathbf{w}^\top \mathbf{x} + b)$$

`predict` is just `predict_proba` compared against a **decision threshold** of 0.5. But 0.5 is only a default - you can move it. If **missing a positive is expensive** (here, missing a malignant tumour), you lower the threshold so borderline cases get flagged: you catch more true positives (higher recall) at the cost of more false alarms (lower precision). This threshold trade-off is the main reason you want probabilities and not just labels.

In [ ]:
# Each row: [P(class 0 = malignant), P(class 1 = benign)]; the two entries sum to 1.
# Values near 0 or 1 mean the model is confident; values near 0.5 mean it is unsure.
probs = model.predict_proba(X_test_s[:5])
print("probabilities for the first 5 samples:")
print(probs.round(3))

In [ ]:
# Demonstrate the decision threshold. P(malignant) is column 0 of predict_proba.
p_malignant = model.predict_proba(X_test_s)[:, 0]

# Compare the default 0.5 threshold with a more cautious 0.3: any tumour whose
# malignant probability exceeds the threshold is flagged malignant (label 0).
for thr in (0.5, 0.3):
    preds_thr = np.where(p_malignant >= thr, 0, 1)   # >= threshold -> malignant (0), else benign (1)
    rec = recall_score(y_test, preds_thr, pos_label=0)   # fraction of malignant tumours caught
    prec = precision_score(y_test, preds_thr, pos_label=0)  # fraction of malignant flags that were right
    print(f"threshold={thr:.1f} -> malignant recall {rec:.3f}, precision {prec:.3f}")
# Lowering the threshold catches more malignant cases (recall up) but raises false alarms (precision down).

## 4. When to Use It

- **Binary or multi-class classification** where you want a simple, fast, interpretable model - it is the standard baseline to try first.
- You need **probabilities**, not just labels, so you can tune the decision threshold to the costs of false positives vs false negatives.
- **Scale your features** (as we did) and raise `max_iter` if the solver fails to converge.
- The **coefficients are interpretable**: each one shows the direction (sign) and strength (magnitude) of a feature's influence on the log-odds of the positive class.
- It draws a **linear** decision boundary. If the classes are separated by a curved boundary, reach for a tree/forest, k-NN, or a neural network instead.